# Evaluator Reliability Stress-Test

This is the part that makes Inference-Lens more than a standard ML pipeline.

LLM-Bar contains 419 response pairs specifically designed to fool automated evaluators. The worse response in each pair was crafted to look better than it is -- grammatically clean, fluent, seemingly on-topic, but actually wrong, unfaithful to the instruction, or subtly misleading. Our models have never seen any of this data.

The question: when the inputs are adversarially constructed, how much does each model's scoring reliability degrade? Which perturbation category causes the most damage? Do certain response quality archetypes get fooled more than others?

The answers are the core finding of this project.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from sklearn.metrics import accuracy_score, roc_auc_score
from inference_lens.data.loader import load_llm_bar
from inference_lens.features.extractor import compute_lexical_features, compute_rouge_l
from inference_lens.stress_test.evaluator import run_stress_test
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../reports/figures', exist_ok=True)

mlflow.set_tracking_uri('../../mlruns')
mlflow.set_experiment('inference-lens')

## Load LLM-Bar

Download LLM-Bar from https://github.com/llm-bar/LLMBar and place the JSON files at `data/raw/llm_bar/`.

The four categories are:
- **neighbor** -- small semantically-similar edits that change the answer
- **natural** -- naturally occurring bad responses that look plausible
- **gpt4_generated** -- adversarial pairs generated by GPT-4
- **manual** -- hand-crafted tricky pairs

In [ ]:
llm_bar = load_llm_bar('../../data/raw/llm_bar')

print(f'LLM-Bar: {len(llm_bar)} pairs')
print(llm_bar['category'].value_counts())
llm_bar.head(3)

## Extract features from LLM-Bar responses

We compute the exact same feature set used during training. The models never saw LLM-Bar, so this is a genuine zero-shot evaluation.

In [ ]:
# LLM-Bar pairs: output1 vs output2, label=1 means output1 is better
# We compute features for each output separately, then let the model score both.
# The model "prefers" whichever output gets a higher chosen-probability.

output1_texts = llm_bar['output1'].tolist()
output2_texts = llm_bar['output2'].tolist()

feat1 = compute_lexical_features(output1_texts)
feat1['rouge_l'] = compute_rouge_l(output1_texts, output2_texts)

feat2 = compute_lexical_features(output2_texts)
feat2['rouge_l'] = compute_rouge_l(output2_texts, output1_texts)

feature_cols = ['token_length', 'type_token_ratio', 'flesch_score', 'rouge_l']
X1 = feat1[feature_cols].values
X2 = feat2[feature_cols].values

print(f'Features computed for {len(X1)} pairs')

## Load trained models and baseline metrics

In [ ]:
scaler    = joblib.load('../../models/scaler.joblib')
logreg    = joblib.load('../../models/logreg.joblib')
xgb_model = joblib.load('../../models/xgboost.joblib')

# load baseline test metrics for degradation calculation
baseline_df = pd.read_parquet('../../data/processed/test_predictions.parquet')
y_baseline  = baseline_df['y_true'].values

logreg_baseline_auc = roc_auc_score(y_baseline, baseline_df['logreg_prob'])
logreg_baseline_acc = accuracy_score(y_baseline, baseline_df['logreg_pred'])
xgb_baseline_auc    = roc_auc_score(y_baseline, baseline_df['xgb_prob'])
xgb_baseline_acc    = accuracy_score(y_baseline, baseline_df['xgb_pred'])

print('Baseline metrics (HH-RLHF test set):')
print(f'  Logistic Regression  AUC: {logreg_baseline_auc:.4f}  Acc: {logreg_baseline_acc:.4f}')
print(f'  XGBoost              AUC: {xgb_baseline_auc:.4f}  Acc: {xgb_baseline_acc:.4f}')

## How pairwise scoring works

For each LLM-Bar pair, we score both outputs individually and let the model vote by comparing scores. If the model assigns a higher chosen-probability to output1, it prefers output1. We compare that preference to the ground truth label.

The false-preference rate is how often the model picks the adversarially preferred (but objectively worse) output.

In [ ]:
def pairwise_preference(model, X1, X2, scale=False, scaler=None):
    """Score each output, return binary preference (1 if output1 preferred)."""
    if scale and scaler is not None:
        X1 = scaler.transform(X1)
        X2 = scaler.transform(X2)
    prob1 = model.predict_proba(X1)[:, 1]
    prob2 = model.predict_proba(X2)[:, 1]
    return (prob1 > prob2).astype(int), prob1, prob2

# ground truth: 1 if output1 is better, 0 if output2 is better
y_llmbar = llm_bar['label'].values.astype(int)
categories = llm_bar['category'].values

logreg_pref, logreg_p1, logreg_p2 = pairwise_preference(logreg, X1, X2, scale=True, scaler=scaler)
xgb_pref,    xgb_p1,    xgb_p2    = pairwise_preference(xgb_model, X1, X2, scale=False)

print(f'LLM-Bar pairwise predictions computed for {len(y_llmbar)} pairs.')

## Run stress-test and measure degradation

In [ ]:
# use preference predictions as our "model output" for stress-test evaluation
# prob score = how confident the model is about its preference
logreg_pref_prob = np.abs(logreg_p1 - logreg_p2)  # confidence margin as proxy
xgb_pref_prob    = np.abs(xgb_p1 - xgb_p2)

# build a fake sklearn-compatible wrapper so we can reuse run_stress_test
class PairwiseWrapper:
    def __init__(self, predictions, probs):
        self._preds = predictions
        self._probs = probs
    def predict(self, X): return self._preds
    def predict_proba(self, X): return np.column_stack([1-self._probs, self._probs])

logreg_results = run_stress_test(
    PairwiseWrapper(logreg_pref, logreg_pref_prob),
    'logreg', X1, y_llmbar, categories,
    baseline_auc=logreg_baseline_auc, baseline_acc=logreg_baseline_acc
)

xgb_results = run_stress_test(
    PairwiseWrapper(xgb_pref, xgb_pref_prob),
    'xgboost', X1, y_llmbar, categories,
    baseline_auc=xgb_baseline_auc, baseline_acc=xgb_baseline_acc
)

stress_results = pd.concat([logreg_results, xgb_results], ignore_index=True)
print('Stress-test complete.')
stress_results

## False-preference rate by perturbation category

Which type of adversarial input does the most damage? The answer tells us something real about what these models are actually sensitive to.

In [ ]:
fpr_pivot = stress_results[stress_results['category'] != 'overall'].pivot(
    index='category', columns='model', values='false_preference_rate'
)

fig, ax = plt.subplots(figsize=(10, 5))
fpr_pivot.plot(kind='bar', ax=ax, color=['steelblue', 'darkorange'], edgecolor='white', width=0.7)
ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='random baseline')
ax.set_title('False-preference rate by LLM-Bar perturbation category')
ax.set_xlabel('Category')
ax.set_ylabel('False-preference rate')
ax.set_ylim(0, 1)
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../../reports/figures/20_false_preference_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## Degradation summary: baseline vs stress-test

In [ ]:
overall = stress_results[stress_results['category'] == 'overall'][[
    'model', 'accuracy', 'auc_roc', 'false_preference_rate', 'acc_degradation', 'auc_degradation'
]].set_index('model')

print('Overall stress-test results vs clean baseline:')
print(overall.round(4).to_string())

In [ ]:
# visualize accuracy degradation
models_list = overall.index.tolist()
baseline_accs = [logreg_baseline_acc, xgb_baseline_acc]
stress_accs   = overall['accuracy'].values

x = np.arange(len(models_list))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, baseline_accs, width, label='Clean baseline (HH-RLHF)', color='steelblue', edgecolor='white')
ax.bar(x + width/2, stress_accs,   width, label='Adversarial (LLM-Bar)',    color='tomato',    edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_title('Accuracy: clean baseline vs adversarial stress-test')
ax.legend()
plt.tight_layout()
plt.savefig('../../reports/figures/21_accuracy_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

## Archetype vulnerability analysis

Do responses that came from certain quality archetypes get fooled more easily? We check whether DBSCAN noise points (the outlier responses from Week 2) are overrepresented in stress-test failures.

In [ ]:
# load the cluster assignments for LLM-Bar outputs
# we'll assign each LLM-Bar output to its nearest K-Means cluster
from inference_lens.features.extractor import compute_embeddings
from sklearn.preprocessing import normalize

print('Computing embeddings for LLM-Bar outputs...')
llmbar_texts = output1_texts + output2_texts
llmbar_emb   = compute_embeddings(llmbar_texts)

# load K-Means cluster centers
from sklearn.cluster import KMeans
import joblib as jl

# refit on existing embeddings to get cluster centers
# (we saved labels but not the model -- next time save the model too)
main_emb = np.load('../../data/processed/embeddings.npy')
features_df = pd.read_parquet('../../data/processed/features_with_clusters.parquet')
optimal_k   = features_df['kmeans_cluster'].nunique()

print(f'Refitting K-Means (K={optimal_k}) to get cluster centers...')
km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
km.fit(main_emb)

llmbar_clusters = km.predict(llmbar_emb)
llmbar_clusters_output1 = llmbar_clusters[:len(output1_texts)]
llmbar_clusters_output2 = llmbar_clusters[len(output1_texts):]
print('Done.')

In [ ]:
# for each pair, check if the model made a false preference and what archetype it came from
archetype_analysis = pd.DataFrame({
    'cluster_output1': llmbar_clusters_output1,
    'cluster_output2': llmbar_clusters_output2,
    'y_true': y_llmbar,
    'logreg_correct': (logreg_pref == y_llmbar).astype(int),
    'xgb_correct': (xgb_pref == y_llmbar).astype(int),
    'category': categories,
})

print('Logistic Regression error rate by preferred output cluster:')
print(archetype_analysis.groupby('cluster_output1')['logreg_correct'].agg(['mean', 'count']).round(3))
print()
print('XGBoost error rate by preferred output cluster:')
print(archetype_analysis.groupby('cluster_output1')['xgb_correct'].agg(['mean', 'count']).round(3))

## DeBERTa stress-test (run after downloading Colab checkpoint)

In [ ]:
checkpoint_path = '../../models/deberta_checkpoint'

if os.path.exists(checkpoint_path):
    from inference_lens.models.train import load_deberta_for_inference
    from transformers import pipeline

    print('Running DeBERTa on LLM-Bar...')
    clf = load_deberta_for_inference(checkpoint_path)

    def deberta_score(texts, batch_size=32):
        scores = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            results = clf(batch, truncation=True, max_length=512)
            scores.extend([r['score'] if r['label'] == 'LABEL_1' else 1 - r['score'] for r in results])
        return np.array(scores)

    deb_p1 = deberta_score(output1_texts)
    deb_p2 = deberta_score(output2_texts)
    deb_pref = (deb_p1 > deb_p2).astype(int)
    deb_pref_prob = np.abs(deb_p1 - deb_p2)

    deberta_preds_df = pd.read_parquet('../../data/processed/deberta_test_predictions.parquet')
    deb_baseline_auc = roc_auc_score(deberta_preds_df['y_true'], deberta_preds_df['deberta_prob'])
    deb_baseline_acc = accuracy_score(deberta_preds_df['y_true'], deberta_preds_df['deberta_pred'])

    deb_results = run_stress_test(
        PairwiseWrapper(deb_pref, deb_pref_prob),
        'deberta', X1, y_llmbar, categories,
        baseline_auc=deb_baseline_auc, baseline_acc=deb_baseline_acc
    )
    stress_results = pd.concat([stress_results, deb_results], ignore_index=True)
    print('DeBERTa stress-test complete.')
else:
    print('DeBERTa checkpoint not found at models/deberta_checkpoint/.')
    print('Run notebook 02 on Colab first, then re-run this cell.')

## Save all stress-test results

In [ ]:
stress_results.to_csv('../../reports/stress_test_results.csv', index=False)
print('Saved reports/stress_test_results.csv')
print()
print('Full results table:')
print(stress_results.to_string(index=False))